# Notebook 1c: Advanced Imputation

**Research-backed 3-stage imputation pipeline:**

| Stage | Method | Paper | Problem Solved |
|-------|--------|-------|----------------|
| 1 | SoftImpute | Mazumder et al., JMLR 2010 | Panel correlations across 80 zones x 8 drugs |
| 2 | HyperImpute | Jarrett et al., ICML 2022 | 5% payer + 3% digital random missing |
| 3 | Smart zero + series mean | Research-backed insight | Cold-start lag nulls (pre-launch periods) |

**Why not GAIN/GRAPE/MIWAE?** Research shows at 3-5% miss rates, iterative methods match deep learning methods. Deep methods win only on large datasets (>50K rows) with heavy missingness.

## Step 1 — Setup

In [ ]:
import sys, warnings
sys.path.append("../03_scripts")
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from pathlib import Path
%matplotlib inline

RAW_DIR       = Path("../01_input/raw")
PROCESSED_DIR = Path("../01_input/processed")
TARGET        = "iqvia_sales_qty_eqv"
print("Setup OK")

## Step 2 — Load Data (with basic cleaning already applied)

In [ ]:
# Load the v2 processed data (DDD normalised, outliers capped)
# If 01b_data_prep_v2 has not been run yet, run it first
try:
    master = pd.read_csv(PROCESSED_DIR / "master_train_v2.csv", low_memory=False)
    test   = pd.read_csv(PROCESSED_DIR / "master_test_v2.csv",  low_memory=False)
    print("Loaded v2 processed data (with basic fixes)")
except:
    master = pd.read_csv(PROCESSED_DIR / "master_train.csv", low_memory=False)
    test   = pd.read_csv(PROCESSED_DIR / "master_test.csv",  low_memory=False)
    print("Loaded v1 processed data (run 01b first for best results)")

gne = master[master["flag_competitor"]=="N"].copy()
gne = gne.sort_values(["product_brand_id","ecosystem_id","date_year_month"])
grp = gne.groupby(["product_brand_id","ecosystem_id"])
print(f"GNE rows: {len(gne):,}")

# Show current missing values
key_cols = ["pct_lives_covered","pct_preferred","digital_impressions",
            "marketing_spend_usd","lag_1","lag_6","lag_12"]
print("\nMissing values in key columns:")
for col in key_cols:
    if col in gne.columns:
        n = gne[col].isna().sum()
        if n > 0: print(f"  {col}: {n:,} nulls ({n/len(gne)*100:.1f}%)")

## Stage 1 — SoftImpute (Panel-Level)

**Why SoftImpute first?**
Our 80 zones × 8 drugs form a low-rank panel — a few shared patterns drive sales across all combinations. SoftImpute finds these patterns and uses them to fill gaps before column-by-column methods. It exploits the fact that Zone 12 and Zone 15 behave similarly for the same drug.

**Paper:** Mazumder, Hastie & Tibshirani, JMLR 2010 — Matrix completion via nuclear norm minimization.

In [ ]:
from fancyimpute import SoftImpute

# Apply SoftImpute to random missing columns only (payer + digital)
RANDOM_MISSING_COLS = [
    "pct_lives_covered", "pct_preferred", "pct_prior_auth_required",
    "pct_step_edit_required", "indication_adjusted_lives",
    "digital_impressions", "marketing_spend_usd", "copay_redemptions",
]

random_cols = [c for c in RANDOM_MISSING_COLS if c in gne.columns]
has_nulls   = [c for c in random_cols if gne[c].isna().sum() > 0]

if has_nulls:
    print(f"Applying SoftImpute to {len(has_nulls)} columns with missing values...")
    X_panel = gne[has_nulls].values.astype(float)
    X_imputed = SoftImpute(verbose=False).fit_transform(X_panel)
    for i, col in enumerate(has_nulls):
        before = gne[col].isna().sum()
        gne[col] = X_imputed[:, i]
        print(f"  {col}: filled {before} nulls using panel correlations")
else:
    print("No random missing values found — SoftImpute not needed")
print("Stage 1 complete.")

## Stage 2 — HyperImpute (Column-Level)

**Why HyperImpute?**
After SoftImpute fills panel-level gaps, HyperImpute handles any remaining missing values by automatically testing multiple models (XGBoost, MICE, MissForest) per column and picking the best one.

**Paper:** Jarrett et al., ICML 2022 — State-of-the-art tabular imputation.

In [ ]:
try:
    from hyperimpute.plugins.imputers import Imputers
    HYPERIMPUTE_AVAILABLE = True
except ImportError:
    HYPERIMPUTE_AVAILABLE = False
    print("HyperImpute not installed — using MissForest fallback")

remaining_nulls = [c for c in random_cols if gne[c].isna().sum() > 0]

if remaining_nulls and HYPERIMPUTE_AVAILABLE:
    print(f"Applying HyperImpute to {len(remaining_nulls)} remaining columns...")
    imputer = Imputers().get("hyperimpute")
    X_rem   = gne[remaining_nulls].copy()
    X_filled = imputer.fit_transform(X_rem)
    for col in remaining_nulls:
        gne[col] = X_filled[col].values
    print("HyperImpute complete.")
elif remaining_nulls:
    # Fallback: MissForest via IterativeImputer + RandomForest
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.ensemble import RandomForestRegressor
    print(f"Using MissForest fallback for {len(remaining_nulls)} columns...")
    mice = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=42),
        max_iter=10, random_state=42
    )
    X_rem   = gne[remaining_nulls].fillna(gne[remaining_nulls].mean())
    X_filled = mice.fit_transform(gne[remaining_nulls].values)
    for i, col in enumerate(remaining_nulls):
        gne[col] = X_filled[:, i]
    print("MissForest imputation complete.")
else:
    print("No remaining nulls after SoftImpute — Stage 2 skipped")

## Stage 3 — Smart Lag Imputation (Cold-Start)

**Research insight:** For pre-launch periods (drug_age < lag period), the true value is 0 — the drug did not exist. This is ground truth, not an estimate. For transitional periods, series mean outperformed KNN in our specific test.

In [ ]:
# Compute raw lags
for lag in [1, 2, 3, 6, 12]:
    gne[f"lag_{lag}"] = grp[TARGET].shift(lag)

# Rolling features
for w in [3, 6]:
    gne[f"roll_mean_{w}"] = grp[TARGET].shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).mean())
    gne[f"roll_std_{w}"] = grp[TARGET].shift(1).transform(
        lambda x: x.rolling(w, min_periods=1).std())

gne["yoy_growth"] = (gne[TARGET] - grp[TARGET].shift(12)) / (
    grp[TARGET].shift(12).abs() + 1e-6)

# Drug age calculation
gne["launch_date_dt"] = pd.to_datetime(gne["launch_date"], errors="coerce")
gne["obs_date"]       = pd.to_datetime(gne["date_year_month"].astype(str), format="%Y%m")
gne["drug_age"]       = (
    (gne["obs_date"].dt.year  - gne["launch_date_dt"].dt.year)*12 +
    (gne["obs_date"].dt.month - gne["launch_date_dt"].dt.month)
).fillna(999)

# Smart fill: pre-launch = 0 (ground truth), rest = series mean
series_means = grp[TARGET].transform("mean")
print("Lag null imputation:")
for lag in [1, 2, 3, 6, 12]:
    col          = f"lag_{lag}"
    pre_launch   = gne["drug_age"] < lag
    n_pre        = (pre_launch & gne[col].isna()).sum()
    n_mean       = (~pre_launch & gne[col].isna()).sum()
    gne[col]     = gne[col].where(gne[col].notna(),
                   np.where(pre_launch, 0.0, series_means))
    print(f"  lag_{lag}: {n_pre} pre-launch → 0 (ground truth) | {n_mean} partial → series mean")

gne["yoy_growth"] = gne["yoy_growth"].fillna(0)
gne = gne.drop(columns=["launch_date_dt","obs_date","drug_age"], errors="ignore")
print("Stage 3 complete.")

## Step 4 — Add Remaining Features

In [ ]:
# Market features
basket = gne.groupby(["date_year_month","ecosystem_id","market_code"])[TARGET].transform("sum")
gne["basket_total_volume"] = basket
gne["market_share"]        = gne[TARGET] / (basket + 1e-8)
gne["competitor_volume"]   = basket - gne[TARGET]

# Payer deltas
gne["delta_pct_preferred"]     = gne["pct_preferred"]     - grp["pct_preferred"].shift(1)
gne["delta_pct_lives_covered"] = gne["pct_lives_covered"] - grp["pct_lives_covered"].shift(1)
gne["access_burden"]           = (gne["pct_prior_auth_required"].fillna(0) +
                                   gne["pct_step_edit_required"].fillna(0)) / 2

# Adstock
DECAY = 0.5
for col in ["rep_calls","digital_impressions","marketing_spend_usd","copay_redemptions"]:
    if col in gne.columns: gne[col] = gne[col].fillna(0)
gne["rep_calls_adstock"] = gne["rep_calls"] + DECAY * grp["rep_calls"].shift(1).fillna(0)
gne["digital_adstock"]   = gne["digital_impressions"].fillna(0) + DECAY * grp["digital_impressions"].shift(1).fillna(0)

# Price
gne["delta_net_price"]     = gne["effective_net_price_per_unit"] - grp["effective_net_price_per_unit"].shift(1)
basket_price               = gne.groupby(["date_year_month","ecosystem_id","market_code"])["effective_net_price_per_unit"].transform("mean")
gne["rel_price_vs_basket"] = gne["effective_net_price_per_unit"] / (basket_price + 1e-8)

# Launch + Fourier
gne["launch_date_dt"]      = pd.to_datetime(gne["launch_date"], errors="coerce")
gne["obs_date"]            = pd.to_datetime(gne["date_year_month"].astype(str), format="%Y%m")
gne["months_since_launch"] = ((gne["obs_date"].dt.year  - gne["launch_date_dt"].dt.year)*12 +
                              (gne["obs_date"].dt.month - gne["launch_date_dt"].dt.month)).fillna(48).clip(lower=0)
gne["flag_cold_start"]     = (gne["months_since_launch"] < 6).astype(int)
gne = gne.drop(columns=["launch_date_dt","obs_date"], errors="ignore")

gne["month_of_year"]  = gne["date_year_month"] % 100
gne["fourier_sin_1"]  = np.sin(2 * np.pi * gne["month_of_year"] / 12)
gne["fourier_cos_1"]  = np.cos(2 * np.pi * gne["month_of_year"] / 12)
gne["fourier_sin_2"]  = np.sin(4 * np.pi * gne["month_of_year"] / 12)
gne["fourier_cos_2"]  = np.cos(4 * np.pi * gne["month_of_year"] / 12)

print(f"Final table: {gne.shape}")
remaining = gne.isna().sum().sum()
print(f"Remaining nulls: {remaining} (should be 0 or very close)")

## Step 5 — Save

In [ ]:
gne.to_csv(PROCESSED_DIR / "master_train_v3.csv", index=False)
print("Saved: 01_input/processed/master_train_v3.csv")
print()
print("=== IMPUTATION SUMMARY ===")
print("Stage 1: SoftImpute  — panel correlations (zones x drugs)")
print("Stage 2: HyperImpute — column-level random missing")
print("Stage 3: Smart zero  — pre-launch lags = 0 (ground truth)")
print("         Series mean — transitional lags")
print("Bonus  : Fourier features — explicit seasonality encoding")
print()
print("Next: train TiDE v5 using master_train_v3.csv")